# Statistical Inference of Typhoon Impacts and Infrastructure Resilience

In this notebook, we perform formal statistical inference to test the validity of our hypotheses regarding the relationship between meteorological intensity, infrastructure investment, and socio-economic outcomes.

The purpose of this notebook is to move beyond exploratory observation and confirm whether the patterns identified in our datasets are statistically significant or merely the result of random chance. This is achieved by applying classical statistical tests to our integrated dataset, which combines typhoon observations with cumulative flood control investment metrics.

The analysis includes:

- Independent T-Tests: To determine if infrastructure planning is responsive to meteorological risks like high-intensity rainfall.

- Chi-Square Tests of Independence: To examine the relationship between "Fiscal Friction" (budgetary inefficiency) and typhoon mortality rates.

- Two-Sample Z-Tests for Proportions: To assess the "Investment Success" of flood control projects and the "False Security" provided during extreme disaster events.

After completing these tests, we will have a mathematically grounded basis to confirm or disprove our conclusions regarding the efficacy of flood control systems and the socio-political factors influencing disaster resilience in the Philippines.

### Import
Start by importing **pandas**, **numpy**, **scipy**, and **statsmodel**.

In [ ]:
# Import relevant python modules
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

## Loading and Merging the Complete Datasets

In this step, we processed datasets by merging the meteorological, infrastructure context, and human impact datasets into a single, unified DataFrame (`df_final`). Because our statistical tests rely on evaluating relationships across different domains—such as comparing infrastructure budget variance with typhoon mortality—all relevant variables must exist within the same table.

The following DataFrames are initialized to facilitate the analysis:
- `df_meteo_infra`: Contains meteorological observations (rainfall, wind speed) enriched with cumulative flood control investment context per province.

- `df_infra`: Contains the individual flood control project details, including the approved budgets and actual contract costs needed for financial accuracy testing.

- `df_impacts`: Contains the standardized human and structural impact records, such as the number of affected persons and casualties.

In [ ]:
# 1. Load the meteorological + infrastructure  dataset
df_meteo_infra = pd.read_csv('../data/merged/typhoon-info-infra-project.csv')

# 2. Load the raw infrastructure projects 
df_infra = pd.read_csv('../data/infra-projects/cleaned_infra_projects.csv')

# 3. Load the impacts dataset (contains Deaths, Affected, Damage, Category)
df_impacts = pd.read_csv('../data/merged/cleaned_typhoon_impacts.csv')

# Rename 'Cyclone Name' to match the 'Typhoon' column in df_meteo_infra
df_impacts = df_impacts.rename(columns={'Cyclone Name': 'Typhoon'})

# Merge them together into df_final based on storm, year, and region
df_final = pd.merge(df_meteo_infra, df_impacts, on=['Typhoon', 'Year', 'Region'], how='inner')

## Step 2: Infrastructure Planning vs. Rainfall Intensity

In this step, we perform an Independent T-test to determine if infrastructure planning is responsive to meteorological risks. We identify "High Intensity Zones" as provinces that frequently experience rainfall exceeding 150 mm and compare their budget allocations against "Low Intensity Zones." By analyzing the Final_Budget_M across these two groups, we can statistically verify whether the government allocates significantly more funds to objectively higher-risk areas, or if the budget distribution is driven by other socio-political factors.

The statistical test is structured as follows:
- Null Hypothesis ($H_0$): There is no significant difference in the mean budget allocation between high-intensity and low-intensity rainfall zones.
- Alternative Hypothesis ($H_a$): High-intensity rainfall zones receive significantly higher budget allocations.

In [ ]:
# Grouping by province to find frequency of high-intensity rain (>150mm)
intensity_counts = df_meteo_infra.groupby('Province')['Max 24-hour Rainfall (mm)'].apply(lambda x: (x > 150).sum())
median_freq = intensity_counts.median()

high_zone_provinces = intensity_counts[intensity_counts > median_freq].index
low_zone_provinces = intensity_counts[intensity_counts <= median_freq].index

# Compare Final_Budget_M for projects in these zones using the raw infrastructure data
high_zone_budgets = df_infra[df_infra['Province'].isin(high_zone_provinces)]['Final_Budget_M']
low_zone_budgets = df_infra[df_infra['Province'].isin(low_zone_provinces)]['Final_Budget_M']

# Using the T-test (equal_var=False) for better accuracy with budget data
t_stat_rain, p_val_rain = stats.ttest_ind(high_zone_budgets, low_zone_budgets, nan_policy='omit', equal_var=False)

print(f"Rainfall vs Planning P-Value: {p_val_rain:.4e}")

Rainfall vs Planning P-Value: 1.0570e-13


### Interpretation: Reject the Null Hypothesis ($H_0$)

The Independent T-test yielded a p-value of **1.0570e-13**. Because this value is far below the standard significance level of 0.05, we **reject the null hypothesis ($H_0$)**.

This highly significant result indicates a clear, mathematical difference in how funds are allocated between high-intensity and low-intensity rainfall zones. This confirms that infrastructure planning is responsive to meteorological risks. While "Political Priority" might influence other areas of governance, the data proves that provinces experiencing heavier, high-intensity rainfall (frequently hitting the 150 mm threshold) consistently receive significantly different budget allocations compared to drier regions.

# Step 3: Testing "Fiscal Friction" and Mortality

In this step, we use a Chi-Square Test of Independence to examine the relationship between financial inefficiencies and human casualties. "Fiscal Friction" is identified when the variance ratio between the approved budget and contract cost exceeds 10%. By creating a contingency table comparing high fiscal friction against occurrences of typhoon mortality, this test evaluates whether budgetary gaps and project inefficiencies are significantly associated with deadlier disaster outcomes.

The statistical test is structured as follows:
- Null Hypothesis ($H_0$): High fiscal friction and high mortality rates are independent.
- Alternative Hypothesis ($H_1$): There is a significant association between high fiscal friction and increased mortality.

In [ ]:
df_final['High_Friction'] = df_final['Variance_Ratio_To_Date'] > 0.10
df_final['High_Mortality'] = df_final['Deaths'] > 0

# Create a 2x2 contingency table
contingency = pd.crosstab(df_final['High_Friction'], df_final['High_Mortality'])
chi2, p_val_friction, dof, ex = stats.chi2_contingency(contingency)

print(f"Fiscal Friction vs Mortality P-Value: {p_val_friction:.4f}")

Fiscal Friction vs Mortality P-Value: 0.6641


### Interpretation: Fail to Reject the Null Hypothesis ($H_0$)

The Chi-Square Test of Independence yielded a p-value of **0.6641**. Because this value is well above our significance level of 0.05, we **fail to reject the null hypothesis ($H_0$)**. 

There is no significant evidence in this dataset to suggest that "Fiscal Friction" (having a budget variance greater than 10%) is associated with higher mortality rates during typhoons. This means we cannot definitively conclude that financial inefficiencies or budgetary gaps directly lead to deadlier disaster outcomes. It suggests that while fiscal friction might delay projects or waste money, it does not necessarily strip the infrastructure of its baseline ability to save lives. It also implies that typhoon casualties are likely driven by other complex factors—such as the sheer severity of the storm, geographical vulnerability, or the presence of early warning evacuation systems—rather than just the financial efficiency of the flood control projects alone.